# Leakage-safe mRMR feature-selection ablation — all original ear-EEG trials

**Purpose:** Compare original F-test and MI top-20 selectors with greedy mRMR top-20, using the exact archived Original80 feature calculation, frozen 3-fold trial-level splits, identical training-only scaling/oversampling and all 315 held-out original trials. This is *not* the separate suspicious-trial deletion experiment. No SingLEM is needed.

**Important:** Run on the original `LMB_05_HW1.zip` dataset. It is not committed to the GitHub repository. All new CSV outputs go under `mrmr_results`, without replacing earlier archived results. mRMR uses the difference objective `MI(feature; emotion) − mean MI(feature; previously chosen features)` (continuous kNN redundancy estimation, `n_neighbors=3`). No hyperparameter tuning on validation results.

## Setup and freeze assumptions

The following cells are copied from the original fixed-fold comparison notebook, including its source archive loader, trial-mean subtraction, filters, handcrafted 80 features, oversampling, learner settings and embedded frozen fold IDs. Only the output directory and the experiment-specific evaluation are changed.

In [ ]:
import time as _stage_time
print('START cell 2',flush=True)
_stage_start=_stage_time.perf_counter()
import importlib.util, subprocess, sys
packages={'numpy':'numpy','scipy':'scipy','pandas':'pandas','sklearn':'scikit-learn','matplotlib':'matplotlib','xgboost':'xgboost'}
missing=[p for m,p in packages.items() if importlib.util.find_spec(m) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import io,json,hashlib,platform,warnings,zipfile
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from scipy.signal import butter,sosfiltfilt,welch,resample_poly,iirnotch,filtfilt
from scipy.integrate import trapezoid
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.feature_selection import f_classif,mutual_info_classif
from sklearn.svm import SVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score,confusion_matrix,ConfusionMatrixDisplay
from xgboost import XGBClassifier
from IPython.display import display
VERSIONS={p:version(p) for p in packages.values()}
print(platform.python_version(),VERSIONS)

print('DONE cell 2:',round(_stage_time.perf_counter()-_stage_start,2),'seconds',flush=True)


In [ ]:
import time as _stage_time
print('START cell 4',flush=True)
_stage_start=_stage_time.perf_counter()
SEED=42; FS=500; CHANNELS=[0,3]; WINDOW=2500; STEP=1250
BANDS={'delta':(1,4),'theta':(4,8),'alpha':(8,12),'beta':(12,30),'gamma':(30,45)}
FILTER_ORDER=4; N_SELECTED=20; BALANCE=True; LABELS=np.array([-1,0,1])
SUBJECT_IDS=None  # None: all 15; [1,7] for a quick affected-subject check
SUSPECTED_TRIALS={(1,6),(7,21)}  # original one-based trial IDs; no automatic additions
SIGNAL_UNIT='uV'  # assumed, not independently verified
REMOVE_TRIAL_CONSTANT=True; ADDITIONAL_NOTCH=False
RUN_SINGLEM=False  # mRMR ablation: no foundation model
SINGLEM_ADDITIONAL_BANDPASS=True
SINGLEM_REPO=Path('/kaggle/working/SingLEM') if Path('/kaggle/working').exists() else Path.cwd()/'SingLEM'
SINGLEM_REF=None
ARCHIVE_PATH=None
DRIVE_ID='1Ma1fHzvhwyQm2zbtYXV8H-SACrmGH88L'
WORK=Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()/'Q2_Fixed_Fold_Comparison'
OUT=WORK/'mrmr_results'; OUT.mkdir(parents=True,exist_ok=True)
if ARCHIVE_PATH is None and not Path('/kaggle/working').exists():
    local_archive=Path.cwd()/'LMB_05_HW1.zip'
    if local_archive.exists():ARCHIVE_PATH=str(local_archive)
config=dict(seed=SEED,fs=FS,channels_zero_based=CHANNELS,window=WINDOW,step=STEP,bands=BANDS,
    n_selected=N_SELECTED,balance=BALANCE,suspected_trials=sorted(SUSPECTED_TRIALS),
    signal_unit_assumed=SIGNAL_UNIT,units_independently_verified=False,
    remove_trial_constant=REMOVE_TRIAL_CONSTANT,additional_notch=ADDITIONAL_NOTCH,
    primary_evaluation='identical nonsuspected validation trials for both training policies',
    secondary_evaluation='suspected validation trials reported separately',versions=VERSIONS)
config['experiment']='F-test vs MI vs mRMR, top20, all original validation trials, fixed folds'
config['mrmr_criterion']='MI relevance minus mean MI redundancy; kNN estimator n_neighbors=3'
print('Results directory:',OUT,flush=True)

print('DONE cell 4:',round(_stage_time.perf_counter()-_stage_start,2),'seconds',flush=True)


In [ ]:
import time as _stage_time
print('START cell 6',flush=True)
_stage_start=_stage_time.perf_counter()
if ARCHIVE_PATH is not None:
    archive=Path(ARCHIVE_PATH)
else:
    candidates=sorted(Path('/kaggle/input').rglob('LMB_05_HW1.zip')) if Path('/kaggle/input').exists() else []
    if len(candidates)>1: raise ValueError('Set ARCHIVE_PATH: multiple ZIPs found.')
    archive=candidates[0] if candidates else WORK/'LMB_05_HW1.zip'
if not archive.exists():
    if importlib.util.find_spec('gdown') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','gdown'])
    import gdown
    gdown.download(id=DRIVE_ID,output=str(archive),quiet=False)
assert zipfile.is_zipfile(archive), 'Expected ZIP, not an HTML access error.'
config['archive_sha256']=hashlib.sha256(archive.read_bytes()).hexdigest()
subjects={}
with zipfile.ZipFile(archive) as z:
    names=z.namelist()
    for name in [n for n in names if n.endswith('.emotion_preprocessed_epoch_InVideo_new.mat')]:
        folder=name.rsplit('/',1)[0]; sid=int(folder.rsplit('/',1)[-1])
        label_files=[n for n in names if n.startswith(folder+'/') and n.endswith('labels.mat')]
        assert len(label_files)==1,(sid,label_files)
        cell=loadmat(io.BytesIO(z.read(name)))['Preprocessed_trials']
        y=loadmat(io.BytesIO(z.read(label_files[0])))['matching_ratings'].ravel()
        assert cell.shape==(1,21) and y.shape==(21,) and np.isin(y,LABELS).all()
        trials=np.stack([np.asarray(t,dtype=np.float64) for t in cell.ravel()])
        assert trials.shape==(21,6,5000),sid
        assert np.isfinite(trials[:,CHANNELS,:]).all(),f'Selected ear contains nonfinite data: {sid}'
        assert sid not in subjects
        subjects[sid]={'trials':trials[:,CHANNELS,:],'y':y.astype(int), 'unused_nonfinite':int((~np.isfinite(trials)).sum())}
assert sorted(subjects)==[1,3,5,7,9,11,13,15,17,19,21,23,27,29,31]
subjects=dict(sorted(subjects.items()))
inventory=pd.DataFrame([dict(subject=sid,trials=21,unused_channel_nonfinite=d['unused_nonfinite'],**{f'label_{c}':int((d['y']==c).sum()) for c in LABELS}) for sid,d in subjects.items()])
display(inventory)
print('Archive SHA-256:',config['archive_sha256'])

assert SIGNAL_UNIT in ['V','mV','uV']
nb,na=iirnotch(50,30,fs=FS)
for sid,d in subjects.items():
    d['original_selected_ears']=d['trials'].copy()
    d['original_trial_numbers']=np.arange(1,22)
    if REMOVE_TRIAL_CONSTANT:d['trials']=d['trials']-d['trials'].mean(axis=-1,keepdims=True)
    if ADDITIONAL_NOTCH:d['trials']=filtfilt(nb,na,d['trials'],axis=-1)
    d['segments']=np.stack([trial[:,start:start+2500] for trial in d['trials'] for start in [0,1250,2500]])
    d['segment_y']=np.repeat(d['y'],3)
    d['trial_ids']=np.repeat(d['original_trial_numbers'],3)
    assert d['segments'].shape==(63,2,2500) and np.isfinite(d['segments']).all()
active_ids=sorted(subjects) if SUBJECT_IDS is None else list(SUBJECT_IDS)
assert active_ids and set(active_ids)<=set(subjects)
assert SUSPECTED_TRIALS <= {(sid,t) for sid in subjects for t in range(1,22)}
print('Original data retained:',len(subjects),'subjects,',sum(len(d['y']) for d in subjects.values()),'trials.',flush=True)

print('DONE cell 6:',round(_stage_time.perf_counter()-_stage_start,2),'seconds',flush=True)


In [ ]:
import time as _stage_time
print('START cell 8',flush=True)
_stage_start=_stage_time.perf_counter()
FROZEN_FOLD_IDS={1: [{'fold': 1, 'train_trials': [2, 3, 4, 5, 8, 11, 13, 14, 15, 16, 17, 18, 20, 21], 'validation_trials': [1, 6, 7, 9, 10, 12, 19]}, {'fold': 2, 'train_trials': [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 18, 19], 'validation_trials': [2, 14, 15, 16, 17, 20, 21]}, {'fold': 3, 'train_trials': [1, 2, 6, 7, 9, 10, 12, 14, 15, 16, 17, 19, 20, 21], 'validation_trials': [3, 4, 5, 8, 11, 13, 18]}], 3: [{'fold': 1, 'train_trials': [3, 6, 7, 8, 9, 10, 12, 14, 15, 16, 17, 18, 19, 20], 'validation_trials': [1, 2, 4, 5, 11, 13, 21]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 7, 11, 13, 15, 16, 18, 19, 20, 21], 'validation_trials': [6, 8, 9, 10, 12, 14, 17]}, {'fold': 3, 'train_trials': [1, 2, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 17, 21], 'validation_trials': [3, 7, 15, 16, 18, 19, 20]}], 5: [{'fold': 1, 'train_trials': [2, 6, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20], 'validation_trials': [1, 3, 4, 5, 7, 12, 21]}, {'fold': 2, 'train_trials': [1, 3, 4, 5, 6, 7, 11, 12, 13, 15, 18, 19, 20, 21], 'validation_trials': [2, 8, 9, 10, 14, 16, 17]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 7, 8, 9, 10, 12, 14, 16, 17, 21], 'validation_trials': [6, 11, 13, 15, 18, 19, 20]}], 7: [{'fold': 1, 'train_trials': [3, 6, 7, 8, 9, 10, 12, 13, 14, 15, 17, 18, 19, 20], 'validation_trials': [1, 2, 4, 5, 11, 16, 21]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 7, 11, 14, 15, 16, 18, 19, 20, 21], 'validation_trials': [6, 8, 9, 10, 12, 13, 17]}, {'fold': 3, 'train_trials': [1, 2, 4, 5, 6, 8, 9, 10, 11, 12, 13, 16, 17, 21], 'validation_trials': [3, 7, 14, 15, 18, 19, 20]}], 9: [{'fold': 1, 'train_trials': [3, 6, 7, 8, 9, 12, 13, 14, 15, 16, 17, 18, 20, 21], 'validation_trials': [1, 2, 4, 5, 10, 11, 19]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 7, 8, 10, 11, 14, 15, 16, 18, 19, 20], 'validation_trials': [3, 6, 9, 12, 13, 17, 21]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 6, 9, 10, 11, 12, 13, 17, 19, 21], 'validation_trials': [7, 8, 14, 15, 16, 18, 20]}], 11: [{'fold': 1, 'train_trials': [2, 3, 4, 6, 8, 9, 10, 12, 15, 17, 18, 19, 20, 21], 'validation_trials': [1, 5, 7, 11, 13, 14, 16]}, {'fold': 2, 'train_trials': [1, 3, 4, 5, 6, 7, 9, 10, 11, 13, 14, 15, 16, 18], 'validation_trials': [2, 8, 12, 17, 19, 20, 21]}, {'fold': 3, 'train_trials': [1, 2, 5, 7, 8, 11, 12, 13, 14, 16, 17, 19, 20, 21], 'validation_trials': [3, 4, 6, 9, 10, 15, 18]}], 13: [{'fold': 1, 'train_trials': [3, 4, 6, 7, 9, 10, 12, 13, 14, 17, 18, 19, 20, 21], 'validation_trials': [1, 2, 5, 8, 11, 15, 16]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 7, 8, 11, 13, 14, 15, 16, 18, 19], 'validation_trials': [6, 9, 10, 12, 17, 20, 21]}, {'fold': 3, 'train_trials': [1, 2, 5, 6, 8, 9, 10, 11, 12, 15, 16, 17, 20, 21], 'validation_trials': [3, 4, 7, 13, 14, 18, 19]}], 15: [{'fold': 1, 'train_trials': [3, 6, 7, 9, 10, 12, 13, 14, 15, 16, 17, 18, 20, 21], 'validation_trials': [1, 2, 4, 5, 8, 11, 19]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 6, 7, 8, 11, 14, 15, 16, 18, 19, 20], 'validation_trials': [3, 9, 10, 12, 13, 17, 21]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 8, 9, 10, 11, 12, 13, 17, 19, 21], 'validation_trials': [6, 7, 14, 15, 16, 18, 20]}], 17: [{'fold': 1, 'train_trials': [1, 3, 5, 6, 7, 8, 12, 13, 14, 16, 17, 19, 20, 21], 'validation_trials': [2, 4, 9, 10, 11, 15, 18]}, {'fold': 2, 'train_trials': [2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 14, 15, 18, 21], 'validation_trials': [1, 5, 13, 16, 17, 19, 20]}, {'fold': 3, 'train_trials': [1, 2, 4, 5, 9, 10, 11, 13, 15, 16, 17, 18, 19, 20], 'validation_trials': [3, 6, 7, 8, 12, 14, 21]}], 19: [{'fold': 1, 'train_trials': [3, 7, 8, 9, 10, 12, 14, 15, 16, 17, 18, 19, 20, 21], 'validation_trials': [1, 2, 4, 5, 6, 11, 13]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 6, 7, 9, 11, 13, 14, 16, 18, 20], 'validation_trials': [8, 10, 12, 15, 17, 19, 21]}, {'fold': 3, 'train_trials': [1, 2, 4, 5, 6, 8, 10, 11, 12, 13, 15, 17, 19, 21], 'validation_trials': [3, 7, 9, 14, 16, 18, 20]}], 21: [{'fold': 1, 'train_trials': [3, 6, 7, 8, 9, 11, 12, 13, 14, 15, 17, 18, 20, 21], 'validation_trials': [1, 2, 4, 5, 10, 16, 19]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 7, 10, 12, 13, 14, 15, 16, 18, 19, 20], 'validation_trials': [3, 6, 8, 9, 11, 17, 21]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 6, 8, 9, 10, 11, 16, 17, 19, 21], 'validation_trials': [7, 12, 13, 14, 15, 18, 20]}], 23: [{'fold': 1, 'train_trials': [1, 3, 6, 7, 9, 10, 12, 14, 15, 16, 17, 19, 20, 21], 'validation_trials': [2, 4, 5, 8, 11, 13, 18]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 8, 9, 11, 12, 13, 14, 15, 16, 18, 20], 'validation_trials': [3, 6, 7, 10, 17, 19, 21]}, {'fold': 3, 'train_trials': [2, 3, 4, 5, 6, 7, 8, 10, 11, 13, 17, 18, 19, 21], 'validation_trials': [1, 9, 12, 14, 15, 16, 20]}], 27: [{'fold': 1, 'train_trials': [3, 5, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21], 'validation_trials': [1, 2, 4, 6, 7, 8, 9]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 6, 7, 8, 9, 10, 16, 17, 18, 19, 21], 'validation_trials': [3, 11, 12, 13, 14, 15, 20]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 6, 7, 8, 9, 11, 12, 13, 14, 15, 20], 'validation_trials': [5, 10, 16, 17, 18, 19, 21]}], 29: [{'fold': 1, 'train_trials': [3, 6, 7, 9, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21], 'validation_trials': [1, 2, 4, 5, 8, 10, 11]}, {'fold': 2, 'train_trials': [1, 2, 4, 5, 6, 7, 8, 10, 11, 14, 16, 18, 19, 21], 'validation_trials': [3, 9, 12, 13, 15, 17, 20]}, {'fold': 3, 'train_trials': [1, 2, 3, 4, 5, 8, 9, 10, 11, 12, 13, 15, 17, 20], 'validation_trials': [6, 7, 14, 16, 18, 19, 21]}], 31: [{'fold': 1, 'train_trials': [1, 2, 5, 6, 7, 9, 10, 12, 14, 15, 16, 17, 20, 21], 'validation_trials': [3, 4, 8, 11, 13, 18, 19]}, {'fold': 2, 'train_trials': [1, 2, 3, 4, 5, 6, 8, 10, 11, 12, 13, 18, 19, 21], 'validation_trials': [7, 9, 14, 15, 16, 17, 20]}, {'fold': 3, 'train_trials': [3, 4, 7, 8, 9, 11, 13, 14, 15, 16, 17, 18, 19, 20], 'validation_trials': [1, 2, 5, 6, 10, 12, 21]}]}
ORIGINAL_FOLD_FILE_SHA256='d657d62bc72561bf7c6abaffcd9df319eab91f61ce129d665ba101ed4adfebc4'

config['original_fold_file_sha256']=ORIGINAL_FOLD_FILE_SHA256
folds={};split_rows=[]
for sid in active_ids:
    d=subjects[sid];seen=[];folds[sid]=[]
    for item in FROZEN_FOLD_IDS[sid]:
        train_ids=np.asarray(item['train_trials'],int);val_ids=np.asarray(item['validation_trials'],int)
        assert not set(train_ids)&set(val_ids)
        assert set(train_ids)|set(val_ids)==set(range(1,22))
        seen.extend(val_ids.tolist())
        suspected_ids={t for s,t in SUSPECTED_TRIALS if s==sid}
        train=np.flatnonzero(np.isin(d['trial_ids'],train_ids))
        train_without=np.flatnonzero(np.isin(d['trial_ids'],[t for t in train_ids if t not in suspected_ids]))
        val_all=np.flatnonzero(np.isin(d['trial_ids'],val_ids))
        val_common=np.flatnonzero(np.isin(d['trial_ids'],[t for t in val_ids if t not in suspected_ids]))
        val_suspect=np.flatnonzero(np.isin(d['trial_ids'],[t for t in val_ids if t in suspected_ids]))
        assert len(train)==42 and len(val_all)==21 and len(val_common)>0
        assert set(val_all)==set(val_common)|set(val_suspect)
        assert not set(val_common)&set(val_suspect)
        assert not set(d['trial_ids'][train_without])&suspected_ids
        fold=dict(fold=item['fold'],train_include=train,train_exclude=train_without,
                  val_all=val_all,val_common=val_common,val_suspect=val_suspect)
        folds[sid].append(fold)
        split_rows.append(dict(subject=sid,fold=item['fold'],
            train_include_trials=json.dumps(train_ids.tolist()),
            train_exclude_trials=json.dumps(np.unique(d['trial_ids'][train_without]).tolist()),
            common_validation_trials=json.dumps(np.unique(d['trial_ids'][val_common]).tolist()),
            suspected_validation_trials=json.dumps(np.unique(d['trial_ids'][val_suspect]).tolist()),
            include_training_windows=len(train),exclude_training_windows=len(train_without),
            common_validation_windows=len(val_common),suspected_validation_windows=len(val_suspect),
            training_data_changed=not np.array_equal(train,train_without),
            **{f'include_train_label_{c}':int((d['segment_y'][train]==c).sum()) for c in LABELS},
            **{f'exclude_train_label_{c}':int((d['segment_y'][train_without]==c).sum()) for c in LABELS},
            **{f'common_val_label_{c}':int((d['segment_y'][val_common]==c).sum()) for c in LABELS}))
    assert sorted(seen)==list(range(1,22))
split_audit=pd.DataFrame(split_rows);split_audit.to_csv(OUT/'fixed_fold_comparison_audit.csv',index=False)
display(split_audit[split_audit.subject.isin([s for s,t in SUSPECTED_TRIALS])])
print('Folds frozen. No split regeneration.',flush=True)

print('DONE cell 8:',round(_stage_time.perf_counter()-_stage_start,2),'seconds',flush=True)


In [ ]:
import time as _stage_time
print('START cell 10',flush=True)
_stage_start=_stage_time.perf_counter()
STARTS=[0,1250,2500]
STAT_NAMES=['mean','std','diff1','norm_diff1','diff2','norm_diff2','relative_power','DE']
FEATURE_NAMES=[f'{ear}_{band}_{stat}' for ear in ['right','left'] for band in BANDS for stat in STAT_NAMES]
SOS={b:butter(FILTER_ORDER,limits,btype='bandpass',fs=FS,output='sos') for b,limits in BANDS.items()}
def integrate_psd(f,p,lo,hi):
    inside=(f>lo)&(f<hi)
    grid=np.r_[lo,f[inside],hi]
    return trapezoid(np.r_[np.interp(lo,f,p),p[inside],np.interp(hi,f,p)],grid)
def handcrafted_features(trials):
    rows=[]
    for trial in trials:
        filtered={b:sosfiltfilt(sos,trial,axis=-1) for b,sos in SOS.items()}
        for start in STARTS:
            row=[]
            for ear in range(2):
                raw=trial[ear,start:start+WINDOW]
                f,p=welch(raw,fs=FS,window='hann',nperseg=1000,noverlap=500,detrend='constant',scaling='density')
                den=integrate_psd(f,p,1,100)
                for band,(lo,hi) in BANDS.items():
                    x=filtered[band][ear,start:start+WINDOW]
                    mu,sd=x.mean(),x.std(ddof=0)
                    z=(x-mu)/sd if sd>0 else np.zeros_like(x)
                    rp=integrate_psd(f,p,lo,hi)/den if den>0 else 0.0
                    row.extend([mu,sd,np.abs(np.diff(x)).mean(),np.abs(np.diff(z)).mean(),
                                np.abs(x[2:]-x[:-2]).mean(),np.abs(z[2:]-z[:-2]).mean(),rp,
                                .5*np.log(2*np.pi*np.e*max(sd**2,1e-30))])
            rows.append(row)
    X=np.asarray(rows)
    assert X.shape==(len(trials)*3,80) and np.isfinite(X).all()
    return X
active_ids=sorted(subjects) if SUBJECT_IDS is None else list(SUBJECT_IDS)
assert active_ids and set(active_ids)<=set(subjects)
for sid in active_ids:
    subjects[sid]['X']=handcrafted_features(subjects[sid]['trials'])
    print('Subject',sid,subjects[sid]['X'].shape)
display(pd.DataFrame(subjects[active_ids[0]]['X'],columns=FEATURE_NAMES).head())

print('DONE cell 10:',round(_stage_time.perf_counter()-_stage_start,2),'seconds',flush=True)


In [ ]:
import time as _stage_time
print('START cell 12',flush=True)
_stage_start=_stage_time.perf_counter()
def select_features(X,y,method,k):
    scores=np.zeros(X.shape[1]); variable=np.flatnonzero(np.var(X,axis=0)>0)
    if method=='MI': scores[variable]=mutual_info_classif(X[:,variable],y,discrete_features=False,random_state=SEED)
    elif method=='F-test':
        with warnings.catch_warnings():
            warnings.simplefilter('ignore'); scores[variable]=f_classif(X[:,variable],y)[0]
    else: raise ValueError(method)
    scores=np.nan_to_num(scores,nan=0.0,posinf=np.finfo(float).max,neginf=0.0)
    return np.argsort(-scores,kind='stable')[:min(k,X.shape[1])],scores

def balanced_rows(y,seed):
    rng=np.random.default_rng(seed); classes,counts=np.unique(y,return_counts=True); rows=[]
    for c,n in zip(classes,counts):
        idx=np.flatnonzero(y==c); rows.extend(idx.tolist())
        if n<counts.max(): rows.extend(rng.choice(idx,size=counts.max()-n,replace=True).tolist())
    return rng.permutation(rows)

def make_model(name,n_classes):
    if name=='SVM': return SVC(C=1.0,kernel='rbf',gamma='scale')
    if name=='LDA': return LinearDiscriminantAnalysis(solver='lsqr',shrinkage='auto')
    if name=='Decision Tree': return DecisionTreeClassifier(max_depth=3,min_samples_leaf=2,random_state=SEED)
    if name=='XGBoost':
        kw=dict(n_estimators=100,max_depth=2,learning_rate=.05,reg_lambda=1.0,subsample=1.0,
                colsample_bytree=1.0,tree_method='hist',n_jobs=1,random_state=SEED,verbosity=0)
        if n_classes>2: kw.update(objective='multi:softprob',num_class=n_classes,eval_metric='mlogloss')
        else: kw.update(objective='binary:logistic',eval_metric='logloss')
        return XGBClassifier(**kw)
    raise ValueError(name)
MODEL_NAMES=['SVM','LDA','Decision Tree','XGBoost']

print('DONE cell 12:',round(_stage_time.perf_counter()-_stage_start,2),'seconds',flush=True)


## mRMR selector

mRMR selection occurs **only inside each outer training fold**, before class oversampling. Relevance is estimated using `mutual_info_classif`; redundancy is estimated using `mutual_info_regression`. Features with zero variance are not eligible. With 20 selections, a candidate is chosen greedily by relevance minus its average redundancy against the already selected features. Overlapping windows remain grouped by trial at the split level.

In [ ]:
from sklearn.feature_selection import mutual_info_regression
from sklearn.metrics import accuracy_score
from scipy import stats

def mrmr_select(X, y, k=20, n_neighbors=3, random_state=SEED):
    """Greedy mRMR-difference selection on *training-only* rows.

    Each feature's relevance is I(feature; class) using mutual_info_classif.
    Redundancy with already selected features is estimated as continuous
    I(feature; selected_feature) via mutual_info_regression.

    Select argmax I(feature; class) - mean_{selected} I(feature; selected).
    The first selected feature maximizes relevance alone.
    """
    X=np.asarray(X,dtype=np.float64)
    y=np.asarray(y)
    if X.ndim!=2 or len(X)!=len(y) or not np.isfinite(X).all():
        raise ValueError('Expected finite 2-D training X with matching labels')
    variable=np.flatnonzero(np.var(X,axis=0)>0)
    if len(variable)==0:raise ValueError('No nonconstant features in training fold')
    V=X[:,variable]
    relevance=np.nan_to_num(
        mutual_info_classif(V,y,discrete_features=False,n_neighbors=n_neighbors,
                            random_state=random_state),
        nan=0.,posinf=0.,neginf=0.
    )
    k=min(int(k),len(variable))
    used=np.zeros(len(variable),dtype=bool)
    redundancy_sums=np.zeros(len(variable),dtype=float)
    picked=[]; audit=[]
    for step in range(k):
        redundancy_mean=redundancy_sums/step if step else np.zeros(len(variable))
        score=relevance-redundancy_mean
        score[used]=-np.inf
        best=int(np.argmax(score))  # deterministic: earliest index wins ties
        j=int(variable[best])
        picked.append(j)
        audit.append(dict(rank=step+1,feature_index=j,
                          relevance_mi=float(relevance[best]),
                          mean_redundancy_mi=float(redundancy_mean[best]),
                          mrmr_score=float(score[best])))
        used[best]=True
        if step+1<k:
            remaining=np.flatnonzero(~used)
            # One call estimates redundancy with the new selected feature
            # for *all* still-eligible candidates (no validation data used).
            redund=mutual_info_regression(V[:,remaining],V[:,best],
                                          discrete_features=False,n_neighbors=n_neighbors,
                                          random_state=random_state)
            redundancy_sums[remaining]+=np.maximum(
                np.nan_to_num(redund,nan=0.,posinf=0.,neginf=0.),0.)
    return np.asarray(picked,dtype=int),audit

# Deterministic synthetic smoke check. This is NOT a dataset result.
_testrng=np.random.default_rng(SEED)
_test_y=np.repeat([-1,0,1],20)
_test_X=np.column_stack((_test_y+_testrng.normal(0,.2,60),
                        _test_y+_testrng.normal(0,.2,60),
                        _testrng.normal(size=60),
                        np.ones(60)))
_test_selection,_test_audit=mrmr_select(_test_X,_test_y,k=3)
assert len(_test_selection)==3 and len(set(_test_selection.tolist()))==3
assert 3 not in _test_selection and [a['rank'] for a in _test_audit]==[1,2,3]
print('Synthetic mRMR smoke checks passed; no real EEG results inferred.')


## Full-validation evaluation, checks and exports

For each subject and fold, use all 14 original training trials and all 7 original validation trials. The primary score is the mean of three fold macro-F1 scores per subject, then the equally weighted mean of 15 subjects, with fixed labels [-1, 0, +1]. A paired bootstrap of subjects is exploratory and does not adjust for model development or multiple comparisons. Because this notebook uses repeated windows for feature ranking (like the original baseline), it does not claim 42 independent training examples.

In [ ]:
# ALL-TRIAL primary comparison: same 315 original trials / 945 validation
# windows across all subjects and folds. No trial exclusion or resplitting.
import time
tic=time.perf_counter()
selector_names=['F-test','MI','mRMR']
out_fold=[];out_predictions=[];out_features=[]
for sid in active_ids:
    d=subjects[sid]
    X=np.asarray(d['X'],dtype=float)
    y=np.asarray(d['segment_y'],dtype=int)
    assert X.shape==(63,80)
    for f in folds[sid]:
        tr=f['train_include']
        va=f['val_all']
        assert len(tr)==42 and len(va)==21
        assert not (set(d['trial_ids'][tr]) & set(d['trial_ids'][va]))
        assert set(d['trial_ids'][va])==set(FROZEN_FOLD_IDS[sid][f['fold']-1]['validation_trials'])
        scaler=StandardScaler().fit(X[tr])
        Xt=scaler.transform(X[tr]);Xv=scaler.transform(X[va])
        encoder=LabelEncoder().fit(y[tr]);yt=encoder.transform(y[tr])
        selections={}
        for method in selector_names:
            if method=='mRMR':
                chosen,audit=mrmr_select(Xt,yt,k=N_SELECTED,n_neighbors=3)
            else:
                chosen,importance=select_features(Xt,yt,method,N_SELECTED)
                audit=[dict(rank=i+1,feature_index=int(j),
                            relevance_mi=(float(importance[j]) if method=='MI' else np.nan),
                            mean_redundancy_mi=np.nan,mrmr_score=float(importance[j]))
                       for i,j in enumerate(chosen)]
            assert len(chosen)==N_SELECTED and len(set(chosen))==N_SELECTED
            selections[method]=chosen
            for item in audit:
                out_features.append(dict(subject=sid,fold=f['fold'],selector=method,
                                         feature=FEATURE_NAMES[item['feature_index']],**item))
        training_rows=balanced_rows(yt,SEED+sid*10+f['fold']) if BALANCE else np.arange(len(tr))
        for method in selector_names:
            chosen=selections[method]
            for model_name in MODEL_NAMES:
                model=make_model(model_name,len(encoder.classes_))
                model.fit(Xt[training_rows][:,chosen],yt[training_rows])
                pred=encoder.inverse_transform(model.predict(Xv[:,chosen]).astype(int))
                out_fold.append(dict(subject=sid,fold=f['fold'],selector=method,
                                     model=model_name,macro_f1=f1_score(y[va],pred,
                                         labels=LABELS,average='macro',zero_division=0),
                                     accuracy=accuracy_score(y[va],pred),
                                     training_trials=14,validation_trials=7,
                                     validation_windows=len(va)))
                for row_index,p in zip(va,pred):
                    out_predictions.append(dict(subject=sid,fold=f['fold'],
                        selector=method,model=model_name,segment=int(row_index),
                        trial=int(d['trial_ids'][row_index]),
                        truth=int(y[row_index]),prediction=int(p)))
    print('Finished subject',sid,';',len(out_fold),'model-fold scores accumulated',flush=True)
fold_df=pd.DataFrame(out_fold)
prediction_df=pd.DataFrame(out_predictions)
features_df=pd.DataFrame(out_features)
assert len(fold_df)==15*3*3*4
assert len(prediction_df)==15*3*3*4*21
assert not prediction_df.duplicated(['subject','selector','model','segment']).any()
assert prediction_df.groupby(['subject','selector','model']).size().eq(63).all()
subject_df=fold_df.groupby(['subject','selector','model'],as_index=False)[['macro_f1','accuracy']].mean()
summary_df=(subject_df.groupby(['selector','model'],as_index=False)
            .agg(mean_macro_f1=('macro_f1','mean'),
                 between_subject_sd=('macro_f1','std'),
                 mean_accuracy=('accuracy','mean'),n_subjects=('subject','nunique')))
summary_df[['mean_macro_f1','between_subject_sd','mean_accuracy']]*=100
summary_df=summary_df.sort_values(['model','mean_macro_f1'],ascending=[True,False])
fold_df.to_csv(OUT/'mrmr_fold_scores.csv',index=False)
prediction_df.to_csv(OUT/'mrmr_predictions.csv',index=False)
features_df.to_csv(OUT/'mrmr_selected_features.csv',index=False)
subject_df.to_csv(OUT/'mrmr_subject_scores.csv',index=False)
summary_df.to_csv(OUT/'mrmr_cohort_summary_percent.csv',index=False)
display(summary_df)
print('Completed:',round(time.perf_counter()-tic,1),'seconds')

# Paired 15-subject uncertainty for mRMR vs each historical selector,
# with all comparisons made on exactly identical original validation folds.
compare=[]
rng=np.random.default_rng(SEED)
for model_name in MODEL_NAMES:
    view=subject_df[subject_df.model==model_name]
    wide=view.pivot(index='subject',columns='selector',values='macro_f1')
    for reference in ['F-test','MI']:
        delta=(wide['mRMR']-wide[reference]).to_numpy()
        boots=rng.choice(delta,size=(10000,len(delta)),replace=True).mean(axis=1)*100
        lo,hi=np.quantile(boots,[.025,.975])
        compare.append(dict(model=model_name,comparison=f'mRMR minus {reference}',
            mean_delta_pp=100*delta.mean(),ci95_low_pp=lo,ci95_high_pp=hi,
            improved_subjects=int((delta>0).sum()),worsened_subjects=int((delta<0).sum()),
            tied_subjects=int((delta==0).sum())))
comparison_df=pd.DataFrame(compare)
comparison_df.to_csv(OUT/'mrmr_paired_bootstrap.csv',index=False)
display(comparison_df)
svm=subject_df[subject_df.model=='SVM'].pivot(index='subject',columns='selector',values='macro_f1')
svm['mrmr_minus_ftest_pp']=(svm['mRMR']-svm['F-test'])*100
svm.to_csv(OUT/'mrmr_SVM_by_subject.csv')
display(svm)
reference_f1=summary_df[(summary_df.selector=='F-test')&(summary_df.model=='SVM')].mean_macro_f1.iloc[0]
mi_f1=summary_df[(summary_df.selector=='MI')&(summary_df.model=='SVM')].mean_macro_f1.iloc[0]
baseline_matches=(abs(reference_f1-35.31)<.03 and abs(mi_f1-33.17)<.03)
print(f'Baseline reference reproduction: F-test/SVM={reference_f1:.4f}%, MI/SVM={mi_f1:.4f}%')
print('Match archived rounded values?',baseline_matches)
if not baseline_matches:
    warnings.warn('Historical baseline check failed. Diagnose environment, data and feature extraction before cross-run interpretation.')
print('Results written to:',OUT)
config.update(mrmr_seed=SEED,mrmr_n_neighbors=3,mrmr_n_selected=N_SELECTED,
    original_full_validation=True,original_baseline_matches_rounded=bool(baseline_matches),
    completed_model_fold_scores=len(fold_df),completed_validation_predictions=len(prediction_df))
(OUT/'mrmr_configuration.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
